# Molecules representations

This notebook tests different representations of molecules. The autoML framework Pycaret is used to quickly test classical machine learning models.

This code is made to be run locally within current repository. If you work in Google Colab, adjust the path to the `processed_df.csv` and `df_fp.csv` file accordingly and install the libraries provided in the requirements.

In [3]:
import pandas as pd
import numpy as np

import pycaret
from pycaret.regression import *

from sklearn.model_selection import cross_val_score, KFold
from sklearn.svm import SVR

## Extended-connectivity fingerprints (ECFPs)

Fingerprints uploads are given in the `Data_processing.ipynb` notebook. Here we use the ready-made file `df_fp.csv`.

In [27]:
df = pd.read_csv('/content/drive/MyDrive/статья/Data/df_fp.csv')

### Model comprasion

In [28]:
df = df.drop('Smiles', axis =1)

In [ ]:
# init the class
exp = RegressionExperiment()

In [ ]:
# check the type of exp
type(exp)

pycaret.regression.oop.RegressionExperiment

In [ ]:
# init setup on exp
exp.setup(df, target = 'pIC50', session_id = 1)

,Description,Value
0,Session id,1
1,Target,pIC50
2,Target type,Regression
3,Original data shape,"(3176, 2049)"
4,Transformed data shape,"(3176, 2049)"
5,Transformed train set shape,"(2223, 2049)"
6,Transformed test set shape,"(953, 2049)"
7,Numeric features,2048
8,Preprocess,True
9,Imputation type,simple


In [ ]:
# compare baseline models
exp.compare_models()

,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE,TT (Sec)
lightgbm,Light Gradient Boosting Machine,0.4457,0.3704,0.6072,0.7099,0.0755,0.0633,2.1170
rf,Random Forest Regressor,0.4455,0.3807,0.6149,0.7026,0.0771,0.0639,22.9400
xgboost,Extreme Gradient Boosting,0.4575,0.3885,0.6214,0.6969,0.0775,0.0648,3.7390
br,Bayesian Ridge,0.4618,0.3881,0.6212,0.6957,0.0772,0.0654,8.3760
knn,K Neighbors Regressor,0.4904,0.4512,0.6685,0.6464,0.0835,0.0704,0.6500
ridge,Ridge Regression,0.4953,0.4665,0.6804,0.6340,0.0839,0.0696,1.3220
huber,Huber Regressor,0.4983,0.4784,0.6894,0.6243,0.0854,0.0700,4.6210
gbr,Gradient Boosting Regressor,0.5282,0.4847,0.6950,0.6206,0.0874,0.0759,5.6990
omp,Orthogonal Matching Pursuit,0.5307,0.4976,0.7039,0.6092,0.0870,0.0746,1.2300
par,Passive Aggressive Regressor,0.5475,0.5504,0.7400,0.5684,0.0911,0.0762,1.4810


Processing:   0%|          | 0/81 [00:00<?, ?it/s]

LGBMRegressor(n_jobs=-1, random_state=1)

In [29]:
X = df.drop(['pIC50'], axis=1)
y = df['pIC50']

In [31]:
model = SVR()

kf = KFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_val_score(model, X, y, cv=kf, scoring='neg_mean_squared_error')

scores_r2 = cross_val_score(model, X, y, cv=kf, scoring='r2')

print(f"MSE: {np.mean(scores):.2f} +/- {np.std(scores):.2f}")
print(f"Mean R2: {np.mean(scores_r2):.2f} +/- {np.std(scores_r2):.2f}")

MSE: -0.34 +/- 0.01
Mean R2: 0.72 +/- 0.01



## Mol2vec

### Descriptor loading

In [ ]:
%pip install git+https://github.com/samoturk/mol2vec

In [ ]:
import sys, os

sys.path.append(os.path.dirname(os.path.dirname(os.getcwd()))+'/mol2vec')

from rdkit import Chem
from rdkit.Chem import PandasTools
from rdkit.Chem.Draw import IPythonConsole
from rdkit.Chem import Draw

from mol2vec.features import mol2alt_sentence, mol2sentence, MolSentence, DfVec, sentences2vec
from gensim.models import word2vec

In [ ]:
df = pd.read_csv('/content/drive/MyDrive/статья/Data/processed_df.csv')

In [ ]:
df = df.drop('IC50', axis=1)

In [ ]:
model = word2vec.Word2Vec.load('/content/drive/MyDrive/статья/Representations/model_300dim.pkl')

In [ ]:
df_mol2 = df

df_mol2['ROMol'] = df_mol2['Smiles'].apply(lambda x: Chem.MolFromSmiles(x))

In [ ]:
df_mol2

,Smiles,pIC50,ROMol
0,Cc1cc(Nc2cc(C(F)(F)F)ccn2)nc(-c2cnc([C@@]3(O)C...,8.560667,<rdkit.Chem.rdchem.Mol object at 0x7d8e2b6b6ce0>
1,Cc1ncc(-c2cn3ccnc3c(Nc3ccc(N4CCN(C5COC5)CC4)cc...,8.207608,<rdkit.Chem.rdchem.Mol object at 0x7d8e2b6b6b20>
2,c1ccc(COc2cccc(Nc3nccc(-c4nccs4)n3)c2)cc1,6.301030,<rdkit.Chem.rdchem.Mol object at 0x7d8e2b6b6d50>
3,c1cncc(Oc2cccc(Nc3nccc(-c4nccs4)n3)c2)c1,6.301030,<rdkit.Chem.rdchem.Mol object at 0x7d8e2b6b6dc0>
4,Cc1cc(Nc2nn(-c3ccc(C(F)(F)F)cc3)c(=O)c3ccccc23...,6.279841,<rdkit.Chem.rdchem.Mol object at 0x7d8e2b6b6e30>
...,...,...,...
3171,CC(C)Cn1cnc(-c2ccnc(Nc3cc(Cl)c4[nH]c(C(=O)N(C)...,6.917574,<rdkit.Chem.rdchem.Mol object at 0x7d8e29daa6c0>
3172,CN1CCC(NC(=O)c2cc3cc(Nc4nccc(-c5cn(C)cn5)n4)cc...,8.244125,<rdkit.Chem.rdchem.Mol object at 0x7d8e29daa730>
3173,Cc1ccc(-c2cnc([C@@]3(O)CC[C@H](C(=O)O)C(C)(C)C...,9.045757,<rdkit.Chem.rdchem.Mol object at 0x7d8e29daa7a0>
3174,NCCCNc2nc(c1cccc(F)c1)cc3ncccc23,6.071000,<rdkit.Chem.rdchem.Mol object at 0x7d8e29daa810>


In [ ]:
df_mol2 ['sentence'] = df_mol2.apply(lambda x: MolSentence(mol2alt_sentence(x['ROMol'], 1)), axis=1)

In [ ]:
vectors = [np.mean([model.wv[word] for word in sentence if word in model.wv], axis=0) for sentence in df_mol2['sentence']]
df_mol2['mol2vec'] = vectors

In [ ]:
df_mol2.columns

Index(['Smiles', 'pIC50', 'ROMol', 'sentence', 'mol2vec'], dtype='object')

In [ ]:
new_mol2vec = np.vstack(df_mol2['mol2vec'].values)
new_mol2vec_columns = [f'2vec_{i}' for i in range(new_mol2vec.shape[1])]
new_mol2vec_df = pd.DataFrame(new_mol2vec, columns=new_mol2vec_columns)

df_mol2vec = pd.concat([df_mol2, new_mol2vec_df], axis=1)

df_mol2vec = df_mol2vec.drop(['ROMol', 'sentence', 'mol2vec'], axis=1)

In [ ]:
df_mol2vec

,Smiles,pIC50,2vec_0,2vec_1,2vec_2,2vec_3,2vec_4,2vec_5,2vec_6,2vec_7,...,2vec_290,2vec_291,2vec_292,2vec_293,2vec_294,2vec_295,2vec_296,2vec_297,2vec_298,2vec_299
0,Cc1cc(Nc2cc(C(F)(F)F)ccn2)nc(-c2cnc([C@@]3(O)C...,8.560667,0.064856,-0.044210,-0.041089,0.133609,-0.031914,-0.018043,-0.237680,0.020456,...,-0.019264,0.208332,0.213050,-0.018601,-0.172787,-0.085745,-0.085349,-0.077389,-0.231701,-0.015070
1,Cc1ncc(-c2cn3ccnc3c(Nc3ccc(N4CCN(C5COC5)CC4)cc...,8.207608,0.119973,-0.148029,-0.045540,0.165872,-0.006626,0.031583,-0.232965,0.011170,...,-0.014664,0.226500,0.172681,-0.072645,-0.185177,-0.050050,-0.122895,-0.030302,-0.226732,-0.033468
2,c1ccc(COc2cccc(Nc3nccc(-c4nccs4)n3)c2)cc1,6.301030,0.024893,-0.083418,-0.045309,0.214921,0.006840,0.032333,-0.241734,-0.022715,...,-0.025218,0.217300,0.244201,-0.037445,-0.117619,-0.057086,-0.078120,-0.049323,-0.210054,-0.028590
3,c1cncc(Oc2cccc(Nc3nccc(-c4nccs4)n3)c2)c1,6.301030,0.050926,-0.099858,-0.052455,0.206724,0.016108,0.032634,-0.231796,-0.017200,...,-0.017454,0.221582,0.254183,-0.044519,-0.114452,-0.054392,-0.087771,-0.046296,-0.227397,-0.020916
4,Cc1cc(Nc2nn(-c3ccc(C(F)(F)F)cc3)c(=O)c3ccccc23...,6.279841,0.081359,-0.079856,-0.076063,0.172600,-0.029412,-0.010877,-0.235642,-0.007948,...,-0.034362,0.187266,0.239657,-0.023037,-0.127927,-0.036033,-0.048713,-0.061949,-0.242218,-0.011565
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3171,CC(C)Cn1cnc(-c2ccnc(Nc3cc(Cl)c4[nH]c(C(=O)N(C)...,6.917574,0.071768,-0.071396,-0.070719,0.118958,0.028896,0.055135,-0.194803,-0.021354,...,-0.003704,0.185389,0.249137,-0.032117,-0.122750,-0.067457,-0.088323,-0.074768,-0.260363,-0.048995
3172,CN1CCC(NC(=O)c2cc3cc(Nc4nccc(-c5cn(C)cn5)n4)cc...,8.244125,0.090950,-0.107925,-0.045017,0.140513,-0.021046,0.011915,-0.201466,-0.005737,...,-0.001768,0.228375,0.217691,-0.072904,-0.146535,-0.069524,-0.122901,-0.066617,-0.246668,-0.029879
3173,Cc1ccc(-c2cnc([C@@]3(O)CC[C@H](C(=O)O)C(C)(C)C...,9.045757,0.032028,-0.053478,-0.044626,0.084859,-0.024854,-0.046218,-0.228670,0.039847,...,-0.010045,0.213080,0.181817,-0.026956,-0.151518,-0.105240,-0.089057,-0.068845,-0.236272,0.012547
3174,NCCCNc2nc(c1cccc(F)c1)cc3ncccc23,6.071000,0.062937,-0.073034,-0.034268,0.182084,0.022170,0.044227,-0.256021,-0.010875,...,-0.027092,0.195613,0.257108,-0.001273,-0.151096,-0.063691,-0.065793,-0.098782,-0.230826,-0.044452


In [ ]:
df_mol2vec.to_csv('/content/drive/MyDrive/статья/Representations/mol2vec_df.csv', index = False)

### Model comprasion

In [ ]:
df_mol2vec = df_mol2vec.drop('Smiles', axis =1)

In [ ]:
exp = RegressionExperiment()

In [ ]:
exp.setup(df_mol2vec, target = 'pIC50', session_id = 1)

,Description,Value
0,Session id,1
1,Target,pIC50
2,Target type,Regression
3,Original data shape,"(3176, 301)"
4,Transformed data shape,"(3176, 301)"
5,Transformed train set shape,"(2223, 301)"
6,Transformed test set shape,"(953, 301)"
7,Numeric features,300
8,Preprocess,True
9,Imputation type,simple


In [ ]:
exp.compare_models()

,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE,TT (Sec)
lightgbm,Light Gradient Boosting Machine,0.4942,0.4565,0.6743,0.6431,0.0842,0.0705,13.3940
et,Extra Trees Regressor,0.4995,0.4803,0.6914,0.6251,0.0864,0.0714,16.4730
rf,Random Forest Regressor,0.5289,0.5092,0.7124,0.6018,0.0891,0.0758,59.7280
knn,K Neighbors Regressor,0.5243,0.5395,0.7327,0.5779,0.0924,0.0767,0.1240
xgboost,Extreme Gradient Boosting,0.5371,0.5479,0.7388,0.5717,0.0920,0.0764,12.1480
br,Bayesian Ridge,0.5550,0.5478,0.7382,0.5697,0.0951,0.0794,0.2060
gbr,Gradient Boosting Regressor,0.5645,0.5504,0.7410,0.5683,0.0919,0.0802,23.3190
huber,Huber Regressor,0.5699,0.5919,0.7680,0.5359,0.0956,0.0820,0.9660
lr,Linear Regression,0.5758,0.6249,0.7846,0.5072,0.0985,0.0824,0.6290
ridge,Ridge Regression,0.6339,0.6773,0.8221,0.4696,0.1019,0.0905,0.1750


Processing:   0%|          | 0/81 [00:00<?, ?it/s]

LGBMRegressor(n_jobs=-1, random_state=1)

In [ ]:
df_mol2vec

,Smiles,pIC50,2vec_0,2vec_1,2vec_2,2vec_3,2vec_4,2vec_5,2vec_6,2vec_7,...,2vec_290,2vec_291,2vec_292,2vec_293,2vec_294,2vec_295,2vec_296,2vec_297,2vec_298,2vec_299
0,Cc1cc(Nc2cc(C(F)(F)F)ccn2)nc(-c2cnc([C@@]3(O)C...,8.560667,0.064856,-0.044210,-0.041089,0.133609,-0.031914,-0.018043,-0.237680,0.020456,...,-0.019264,0.208332,0.213050,-0.018601,-0.172787,-0.085745,-0.085349,-0.077389,-0.231701,-0.015070
1,Cc1ncc(-c2cn3ccnc3c(Nc3ccc(N4CCN(C5COC5)CC4)cc...,8.207608,0.119973,-0.148029,-0.045540,0.165872,-0.006626,0.031583,-0.232965,0.011170,...,-0.014664,0.226500,0.172681,-0.072645,-0.185177,-0.050050,-0.122895,-0.030302,-0.226732,-0.033468
2,c1ccc(COc2cccc(Nc3nccc(-c4nccs4)n3)c2)cc1,6.301030,0.024893,-0.083418,-0.045309,0.214921,0.006840,0.032333,-0.241734,-0.022715,...,-0.025218,0.217300,0.244201,-0.037445,-0.117619,-0.057086,-0.078120,-0.049323,-0.210054,-0.028590
3,c1cncc(Oc2cccc(Nc3nccc(-c4nccs4)n3)c2)c1,6.301030,0.050926,-0.099858,-0.052455,0.206724,0.016108,0.032634,-0.231796,-0.017200,...,-0.017454,0.221582,0.254183,-0.044519,-0.114452,-0.054392,-0.087771,-0.046296,-0.227397,-0.020916
4,Cc1cc(Nc2nn(-c3ccc(C(F)(F)F)cc3)c(=O)c3ccccc23...,6.279841,0.081359,-0.079856,-0.076063,0.172600,-0.029412,-0.010877,-0.235642,-0.007948,...,-0.034362,0.187266,0.239657,-0.023037,-0.127927,-0.036033,-0.048713,-0.061949,-0.242218,-0.011565
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3171,CC(C)Cn1cnc(-c2ccnc(Nc3cc(Cl)c4[nH]c(C(=O)N(C)...,6.917574,0.071768,-0.071396,-0.070719,0.118958,0.028896,0.055135,-0.194803,-0.021354,...,-0.003704,0.185389,0.249137,-0.032117,-0.122750,-0.067457,-0.088323,-0.074768,-0.260363,-0.048995
3172,CN1CCC(NC(=O)c2cc3cc(Nc4nccc(-c5cn(C)cn5)n4)cc...,8.244125,0.090950,-0.107925,-0.045017,0.140513,-0.021046,0.011915,-0.201466,-0.005737,...,-0.001768,0.228375,0.217691,-0.072904,-0.146535,-0.069524,-0.122901,-0.066617,-0.246668,-0.029879
3173,Cc1ccc(-c2cnc([C@@]3(O)CC[C@H](C(=O)O)C(C)(C)C...,9.045757,0.032028,-0.053478,-0.044626,0.084859,-0.024854,-0.046218,-0.228670,0.039847,...,-0.010045,0.213080,0.181817,-0.026956,-0.151518,-0.105240,-0.089057,-0.068845,-0.236272,0.012547
3174,NCCCNc2nc(c1cccc(F)c1)cc3ncccc23,6.071000,0.062937,-0.073034,-0.034268,0.182084,0.022170,0.044227,-0.256021,-0.010875,...,-0.027092,0.195613,0.257108,-0.001273,-0.151096,-0.063691,-0.065793,-0.098782,-0.230826,-0.044452


In [ ]:
X = df_mol2vec.drop(['Smiles', 'pIC50'], axis=1)
y = df_mol2vec['pIC50']

In [ ]:
model = SVR()

kf = KFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_val_score(model, X, y, cv=kf, scoring='neg_mean_squared_error')

In [ ]:
scores_r2 = cross_val_score(model, X, y, cv=kf, scoring='r2')

In [ ]:
print(f"MSE: {np.mean(scores):.2f} +/- {np.std(scores):.2f}")
print(f"Среднее R2: {np.mean(scores_r2):.2f} +/- {np.std(scores_r2):.2f}")

MSE: -0.66 +/- 0.03
Среднее R2: 0.47 +/- 0.02


## MACCS fingerprints

In [ ]:
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem import MACCSkeys

In [ ]:
df = pd.read_csv('/content/drive/MyDrive/статья/Data/processed_df.csv')

### Descriptor loading

In [ ]:
def get_maccs_fp(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is not None:
        return MACCSkeys.GenMACCSKeys(mol)
    return None

In [ ]:
df['MACCS_FP'] = df['Smiles'].apply(get_maccs_fp)

df['MACCS_FP_BitString'] = df['MACCS_FP'].apply(lambda x: x.ToBitString() if x is not None else None)

In [ ]:
fp_df = df['MACCS_FP_BitString'].str.split('', expand=True).iloc[:, 1:-1]

df_MACCS = pd.concat([df, fp_df], axis=1)

df_MACCS = df_MACCS.drop(['MACCS_FP', 'MACCS_FP_BitString'], axis=1)

In [ ]:
df_MACCS = df_MACCS.drop(['IC50'], axis = 1)

In [ ]:
df_MACCS.to_csv('/content/drive/MyDrive/статья/Representations/MACCS_df.csv', index=False)

### Model comprasion

In [ ]:
df_MACCS = df_MACCS.drop('Smiles', axis =1)

In [ ]:
# init the class
exp = RegressionExperiment()

In [ ]:
# init setup on exp
exp.setup(df_MACCS, target = 'pIC50', session_id = 1)

,Description,Value
0,Session id,1
1,Target,pIC50
2,Target type,Regression
3,Original data shape,"(3176, 168)"
4,Transformed data shape,"(3176, 168)"
5,Transformed train set shape,"(2223, 168)"
6,Transformed test set shape,"(953, 168)"
7,Categorical features,167
8,Preprocess,True
9,Imputation type,simple


In [ ]:
# compare baseline models
exp.compare_models()

,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE,TT (Sec)
lightgbm,Light Gradient Boosting Machine,0.5042,0.4680,0.6818,0.6349,0.0850,0.0720,0.8860
rf,Random Forest Regressor,0.5119,0.5014,0.7052,0.6092,0.0881,0.0733,0.3560
knn,K Neighbors Regressor,0.5408,0.5634,0.7471,0.5597,0.0937,0.0781,0.1850
gbr,Gradient Boosting Regressor,0.5814,0.5839,0.7623,0.5439,0.0947,0.0829,0.2350
br,Bayesian Ridge,0.6482,0.7094,0.8402,0.4455,0.1043,0.0926,0.1860
ridge,Ridge Regression,0.6531,0.7307,0.8525,0.4287,0.1061,0.0934,0.1730
huber,Huber Regressor,0.6489,0.7483,0.8622,0.4149,0.1075,0.0934,0.1870
et,Extra Trees Regressor,0.6110,0.7722,0.8767,0.3943,0.1081,0.0860,0.4260
ada,AdaBoost Regressor,0.7318,0.8409,0.9156,0.3422,0.1126,0.1034,0.2310
dt,Decision Tree Regressor,0.6358,0.8452,0.9173,0.3375,0.1133,0.0896,0.1740


LGBMRegressor(n_jobs=-1, random_state=1)

In [ ]:
X = df_MACCS.drop(['Smiles', 'pIC50'], axis=1)
y = df_MACCS['pIC50']

In [ ]:
model = SVR()

kf = KFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_val_score(model, X, y, cv=kf, scoring='neg_mean_squared_error')

In [ ]:
scores_r2 = cross_val_score(model, X, y, cv=kf, scoring='r2')

In [ ]:
print(f"MSE: {np.mean(scores):.2f} +/- {np.std(scores):.2f}")
print(f"Среднее R2: {np.mean(scores_r2):.2f} +/- {np.std(scores_r2):.2f}")

MSE: -0.49 +/- 0.02
Среднее R2: 0.60 +/- 0.01


## MinHashed Atom Pair (MAP4)

In [ ]:
from map4 import MAP4
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem
import numpy as np

### Descriptor loading

In [ ]:
df = pd.read_csv('/content/drive/MyDrive/статья/Data/processed_df.csv')

In [ ]:
map4 = MAP4(
    dimensions=2048,
    radius=2,
    include_duplicated_shingles=False,
)

In [ ]:
# Function for calculating the MAP4 fingerprint
def calculate_map4(smiles):
    mol = Chem.MolFromSmiles(smiles)
    return map4.calculate(mol)

In [ ]:
fingerprints = df['Smiles'].apply(calculate_map4)

MAP4_fingerprints = pd.DataFrame(fingerprints.tolist(), index=df.index)
MAP4_fingerprints.columns = [f'MAP4_{i}' for i in range(2048)]

df_MAP4 = pd.concat([df, MAP4_fingerprints], axis=1)

In [ ]:
df_MAP4

,Smiles,IC50,pIC50,MAP4_0,MAP4_1,MAP4_2,MAP4_3,MAP4_4,MAP4_5,MAP4_6,...,MAP4_2038,MAP4_2039,MAP4_2040,MAP4_2041,MAP4_2042,MAP4_2043,MAP4_2044,MAP4_2045,MAP4_2046,MAP4_2047
0,Cc1cc(Nc2cc(C(F)(F)F)ccn2)nc(-c2cnc([C@@]3(O)C...,2.750000,8.560667,0,1,1,1,1,0,0,...,0,1,0,1,0,1,1,0,1,0
1,Cc1ncc(-c2cn3ccnc3c(Nc3ccc(N4CCN(C5COC5)CC4)cc...,6.200000,8.207608,1,0,0,0,0,0,0,...,1,0,1,1,0,0,0,0,0,0
2,c1ccc(COc2cccc(Nc3nccc(-c4nccs4)n3)c2)cc1,500.000000,6.301030,1,0,0,0,1,0,0,...,0,1,0,0,0,0,0,0,0,0
3,c1cncc(Oc2cccc(Nc3nccc(-c4nccs4)n3)c2)c1,500.000000,6.301030,1,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
4,Cc1cc(Nc2nn(-c3ccc(C(F)(F)F)cc3)c(=O)c3ccccc23...,525.000000,6.279841,0,0,0,1,1,0,0,...,0,1,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3171,CC(C)Cn1cnc(-c2ccnc(Nc3cc(Cl)c4[nH]c(C(=O)N(C)...,120.900000,6.917574,0,0,1,1,0,1,0,...,0,0,0,0,1,1,0,0,0,0
3172,CN1CCC(NC(=O)c2cc3cc(Nc4nccc(-c5cn(C)cn5)n4)cc...,5.700000,8.244125,0,0,1,0,0,1,0,...,0,0,0,0,0,1,0,0,0,0
3173,Cc1ccc(-c2cnc([C@@]3(O)CC[C@H](C(=O)O)C(C)(C)C...,0.900000,9.045757,0,0,0,0,0,1,0,...,0,1,0,0,0,1,0,0,0,1
3174,NCCCNc2nc(c1cccc(F)c1)cc3ncccc23,849.180475,6.071000,0,0,0,0,1,0,0,...,0,0,0,0,0,1,0,0,0,0


In [ ]:
df_MAP4.to_csv('/content/drive/MyDrive/статья/Representations/MAP4_df.csv', index=False)

### Model comprasion

In [ ]:
import pycaret
from pycaret.regression import *

In [ ]:
# init the class
exp = RegressionExperiment()

In [ ]:
df_MAP4 = df_MAP4.drop(['Smiles', 'IC50'], axis =1)

In [ ]:
# init setup on exp
exp.setup(df_MAP4, target='pIC50', session_id=1,
          fold=5)

,Description,Value
0,Session id,1
1,Target,pIC50
2,Target type,Regression
3,Original data shape,"(3176, 2049)"
4,Transformed data shape,"(3176, 2049)"
5,Transformed train set shape,"(2223, 2049)"
6,Transformed test set shape,"(953, 2049)"
7,Numeric features,2048
8,Preprocess,True
9,Imputation type,simple


In [ ]:
# compare baseline models
exp.compare_models()

,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE,TT (Sec)
lightgbm,Light Gradient Boosting Machine,0.4950,0.4604,0.6767,0.6421,0.0850,0.0709,1.3660
br,Bayesian Ridge,0.5108,0.4675,0.6832,0.6346,0.0852,0.0726,2.7580
knn,K Neighbors Regressor,0.5034,0.4753,0.6888,0.6284,0.0861,0.0719,0.1500
gbr,Gradient Boosting Regressor,0.5246,0.4962,0.7037,0.6129,0.0884,0.0752,2.0560
rf,Random Forest Regressor,0.5212,0.5037,0.7082,0.6080,0.0888,0.0747,5.0420
ada,AdaBoost Regressor,0.6274,0.6487,0.8041,0.4950,0.0997,0.0887,2.3280
omp,Orthogonal Matching Pursuit,0.6150,0.6637,0.8146,0.4802,0.1014,0.0873,0.4300
et,Extra Trees Regressor,0.6762,0.9119,0.9540,0.2855,0.1178,0.0959,7.6980
huber,Huber Regressor,0.7195,0.9430,0.9695,0.2595,0.1232,0.1001,0.9920
lar,Least Angle Regression,0.7469,0.9668,0.9791,0.2485,0.1215,0.1060,1.0200


LGBMRegressor(n_jobs=-1, random_state=1)

In [ ]:
X = df_MAP4.drop(['Smiles', 'pIC50', 'IC50'], axis=1)
y = df_MAP4['pIC50']

In [ ]:
model = SVR()

kf = KFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_val_score(model, X, y, cv=kf, scoring='neg_mean_squared_error')

scores_r2 = cross_val_score(model, X, y, cv=kf, scoring='r2')

print(f"MSE: {np.mean(scores):.2f} +/- {np.std(scores):.2f}")
print(f"Среднее R2: {np.mean(scores_r2):.2f} +/- {np.std(scores_r2):.2f}")

MSE: -0.38 +/- 0.02
Среднее R2: 0.69 +/- 0.01


## PubChem fingerprints

### Descriptor loading

In [ ]:
pip install pubchempy

In [ ]:
import deepchem as dc

In [ ]:
df = pd.read_csv('/content/drive/MyDrive/статья/Data/processed_df.csv')

In [ ]:
featurizer = dc.feat.PubChemFingerprint()
features = featurizer.featurize(df['Smiles'])

In [ ]:
fp_df = pd.DataFrame(features)

In [ ]:
def convert_to_array(array_str):
  if isinstance(array_str, str):
    return np.array(ast.literl_eval(array_str.replace('\n', '')))
  return array_str

In [ ]:
fp_df[0] = fp_df[0].apply(convert_to_array)

In [ ]:
features_df = pd.DataFrame(fp_df[0].tolist(), index=fp_df.index)
features_df.columns = [f'fp_{i+1}' for i in range(features_df.shape[1])]

In [ ]:
df_pubchem = pd.concat([df, features_df], axia = 1)

In [ ]:
df_pubchem.to_csv('/content/drive/MyDrive/статья/df_pubchem.csv', index = False)

### Model comprasion

In [20]:
df_pubchem = pd.read_csv('/content/drive/MyDrive/статья/df_pubchem.csv')

In [5]:
# init the class
exp = RegressionExperiment()

In [21]:
df_pubchem = df_pubchem.drop(['Smiles', 'IC50'], axis =1)

In [8]:
# init setup on exp
exp.setup(df_pubchem, target='pIC50', session_id=1,
          fold=5)

,Description,Value
0,Session id,1
1,Target,pIC50
2,Target type,Regression
3,Original data shape,"(3176, 882)"
4,Transformed data shape,"(3176, 882)"
5,Transformed train set shape,"(2223, 882)"
6,Transformed test set shape,"(953, 882)"
7,Numeric features,881
8,Rows with missing values,0.4%
9,Preprocess,True


In [9]:
# compare baseline models
exp.compare_models()

,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE,TT (Sec)
lightgbm,Light Gradient Boosting Machine,0.4798,0.4357,0.6586,0.6597,0.0824,0.0685,1.1280
xgboost,Extreme Gradient Boosting,0.4714,0.4361,0.6591,0.6595,0.0824,0.0670,1.1580
rf,Random Forest Regressor,0.4745,0.4465,0.6669,0.6512,0.0834,0.0679,3.6600
knn,K Neighbors Regressor,0.5237,0.5242,0.7232,0.5902,0.0907,0.0755,0.2720
gbr,Gradient Boosting Regressor,0.5477,0.5374,0.7321,0.5811,0.0914,0.0783,1.8140
br,Bayesian Ridge,0.5506,0.5376,0.7323,0.5807,0.0911,0.0785,0.5780
ridge,Ridge Regression,0.5596,0.5657,0.7504,0.5588,0.0937,0.0797,0.3100
huber,Huber Regressor,0.5587,0.5777,0.7588,0.5489,0.0945,0.0799,1.2380
omp,Orthogonal Matching Pursuit,0.5779,0.5857,0.7642,0.5433,0.0949,0.0821,0.2080
et,Extra Trees Regressor,0.5472,0.6310,0.7925,0.5069,0.0982,0.0778,5.1120


Processing:   0%|          | 0/81 [00:00<?, ?it/s]

LGBMRegressor(n_jobs=-1, random_state=1)

In [22]:
indices_to_remove = [2778, 2779, 2780, 2781, 2784, 2785, 2842, 2853, 2906, 2908, 2909, 2910]
df_pubchem = df_pubchem.drop(indices_to_remove, axis=0, errors='ignore')

In [23]:
X = df_pubchem.drop(['pIC50'], axis=1)
y = df_pubchem['pIC50']

In [26]:
model = SVR()

kf = KFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_val_score(model, X, y, cv=kf, scoring='neg_mean_squared_error')

scores_r2 = cross_val_score(model, X, y, cv=kf, scoring='r2')

print(f"MSE: {np.mean(scores):.2f} +/- {np.std(scores):.2f}")
print(f"Mean R2: {np.mean(scores_r2):.2f} +/- {np.std(scores_r2):.2f}")

MSE: -0.47 +/- 0.04
Среднее R2: 0.62 +/- 0.02
